### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Sarthak.Somani\AppData\Local\Temp\ipykernel_9252\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: Tally Prime to Excel Reporting Pipeline.pdf
  ✓ Loaded 11 pages

Total documents loaded: 11


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-10-22T15:19:05+05:30', 'title': 'Tally Prime to Excel Reporting Pipeline', 'moddate': '2026-10-22T15:19:05+05:30', 'source': '..\\data\\pdf_files\\Tally Prime to Excel Reporting Pipeline.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Tally Prime to Excel Reporting Pipeline.pdf', 'file_type': 'pdf'}, page_content="Tally Prime to Excel Reporting Pipeline \nSolution Design  \nPrepared October 2026 | For internal review  \n \nExecutive Summary \nFinance reports are currently produced by manually exporting data from TallyPrime. This is slow, error-\nprone and hard to repeat. This document proposes an automated pipeline that keeps a reporting \ndatabase continuously in sync with Tally and lets Excel read from it, so that reports refresh with one click \nand always agree with the books. \nWhat changes \n• Data moves from Tally to a database automatically, ev

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 11 documents into 27 chunks

Example chunk:
Content: Tally Prime to Excel Reporting Pipeline 
Solution Design  
Prepared October 2026 | For internal review  
 
Executive Summary 
Finance reports are currently produced by manually exporting data from Tal...
Metadata: {'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-10-22T15:19:05+05:30', 'title': 'Tally Prime to Excel Reporting Pipeline', 'moddate': '2026-10-22T15:19:05+05:30', 'source': '..\\data\\pdf_files\\Tally Prime to Excel Reporting Pipeline.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Tally Prime to Excel Reporting Pipeline.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-10-22T15:19:05+05:30', 'title': 'Tally Prime to Excel Reporting Pipeline', 'moddate': '2026-10-22T15:19:05+05:30', 'source': '..\\data\\pdf_files\\Tally Prime to Excel Reporting Pipeline.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Tally Prime to Excel Reporting Pipeline.pdf', 'file_type': 'pdf'}, page_content="Tally Prime to Excel Reporting Pipeline \nSolution Design  \nPrepared October 2026 | For internal review  \n \nExecutive Summary \nFinance reports are currently produced by manually exporting data from TallyPrime. This is slow, error-\nprone and hard to repeat. This document proposes an automated pipeline that keeps a reporting \ndatabase continuously in sync with Tally and lets Excel read from it, so that reports refresh with one click \nand always agree with the books. \nWhat changes \n• Data moves from Tally to a database automatically, ev

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Sarthak.Somani\ytrag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2422.85it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Sarthak.Somani\AppData\Local\Temp\ipykernel_9252\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 27


In [9]:
chunks

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-10-22T15:19:05+05:30', 'title': 'Tally Prime to Excel Reporting Pipeline', 'moddate': '2026-10-22T15:19:05+05:30', 'source': '..\\data\\pdf_files\\Tally Prime to Excel Reporting Pipeline.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Tally Prime to Excel Reporting Pipeline.pdf', 'file_type': 'pdf'}, page_content="Tally Prime to Excel Reporting Pipeline \nSolution Design  \nPrepared October 2026 | For internal review  \n \nExecutive Summary \nFinance reports are currently produced by manually exporting data from TallyPrime. This is slow, error-\nprone and hard to repeat. This document proposes an automated pipeline that keeps a reporting \ndatabase continuously in sync with Tally and lets Excel read from it, so that reports refresh with one click \nand always agree with the books. \nWhat changes \n• Data moves from Tally to a database automatically, ev

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 27 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.49s/it]


Generated embeddings with shape: (27, 384)
Adding 27 documents to vector store...
Successfully added 27 documents to vector store
Total documents in collection: 54


### Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("How does the pipeline handle vouchers that are deleted in Tally, and why can’t the AlterID filter detect them?")

Retrieving documents for query: 'How does the pipeline handle vouchers that are deleted in Tally, and why can’t the AlterID filter detect them?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.37it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_2b1dadec_5',
  'content': 'Amounts are normalised on load (debit positive, credit negative) and verified against a known \nvoucher. \n• Cancelled vouchers are kept, flagged, and excluded in the views, preserving the audit trail. \n \nStep 3: Extract Data from Tally \nTallyPrime has a built-in HTTP and XML server. The agent sends a request defined in Tally Definition \nLanguage (TDL) and receives only the fields it needs. \nEnabling the server: in TallyPrime, go to F1 (Help), then Settings, then Connectivity, then Client/Server \nconfiguration. Set TallyPrime to act as Server (or Both) on port 9000. The company must be open. \nIncremental extraction \nEvery create, edit or delete in Tally increases its AlterID counter. The agent stores the last AlterID it \nprocessed and asks Tally only for records above that value. This keeps each run fast, typically seconds, \ninstead of re-reading the whole company. \nCollection: Voucher \nFilter:     $AlterID > {last_alter_id}',
  'meta

In [14]:
rag_retriever.retrieve("Why does the receivables and payables ageing view use a nightly snapshot of Tally’s own reports instead of recomputing outstandings from vouchers?")


Retrieving documents for query: 'Why does the receivables and payables ageing view use a nightly snapshot of Tally’s own reports instead of recomputing outstandings from vouchers?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4adef66f_9',
  'content': "View Contents \nledger_monthly Net movement per ledger per month \nreceivables and payables ageing Built from the nightly snapshot of Tally's own Bills \nReceivable and Bills Payable reports \nGST summary Output and input tax by period (to be defined with the \ntax team) \nstock summary Quantity and value by item and godown \nWhy ageing uses a snapshot. Bill-wise settlements, advances and on-account entries make recomputing \noutstandings from vouchers error-prone. Using Tally's own report guarantees the ageing matches what \nthe accountant sees in Tally. \nReconciliation control. Each night the pipeline stores Tally's trial balance and compares it with the totals \nin the database. Any difference triggers an email alert. This is the main safeguard against silent data \ndrift. \n \nStep 6: Build the Excel Reports \n1. Connect. In Excel, choose Data, Get Data, From Database, and select the views (not the tables).",
  'metadata': {'creator': 'Micro

In [15]:
rag_retriever.retrieve("Which database options does the document compare for the reporting database, and which one does it recommend?")


Retrieving documents for query: 'Which database options does the document compare for the reporting database, and which one does it recommend?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.14it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_5fc423bd_1',
  'content': 'report. \n• Business logic (sales register, ageing, monthly trends) lives in one place and is reused by every \nreport. \nWhat stays the same \n• Tally remains the single system of record. The pipeline only reads from it. \n• Finance users keep working in Excel, with no new tool to learn. \nKey benefits \nBenefit Outcome \nTime saved Report preparation drops from hours of export and clean-up to a single \nrefresh \nAccuracy Totals are reconciled to Tally automatically, and silent drift is caught early \nConsistency One set of definitions feeds every report and every user \nScalability Adding a report means adding a view, not rebuilding an export process \nFuture-ready The same database can later feed Power BI Service or other tools without \nrework',
  'metadata': {'file_type': 'pdf',
   'page': 0,
   'creator': 'Microsoft® Word LTSC',
   'producer': 'Microsoft® Word LTSC',
   'creationdate': '2026-10-22T15:19:05+05:30',
   'source': '..\\data\\p

### RAG Pipeline- VectorDB To LLM Output Generation

In [50]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

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


In [51]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [52]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-120b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [53]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-120b
Groq LLM initialized successfully!


In [54]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("How does the pipeline handle vouchers that are deleted in Tally, and why can’t the AlterID filter detect them?")

Retrieving documents for query: 'How does the pipeline handle vouchers that are deleted in Tally, and why can’t the AlterID filter detect them?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.73it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_2b1dadec_5',
  'content': 'Amounts are normalised on load (debit positive, credit negative) and verified against a known \nvoucher. \n• Cancelled vouchers are kept, flagged, and excluded in the views, preserving the audit trail. \n \nStep 3: Extract Data from Tally \nTallyPrime has a built-in HTTP and XML server. The agent sends a request defined in Tally Definition \nLanguage (TDL) and receives only the fields it needs. \nEnabling the server: in TallyPrime, go to F1 (Help), then Settings, then Connectivity, then Client/Server \nconfiguration. Set TallyPrime to act as Server (or Both) on port 9000. The company must be open. \nIncremental extraction \nEvery create, edit or delete in Tally increases its AlterID counter. The agent stores the last AlterID it \nprocessed and asks Tally only for records above that value. This keeps each run fast, typically seconds, \ninstead of re-reading the whole company. \nCollection: Voucher \nFilter:     $AlterID > {last_alter_id}',
  'meta

### Integration Vectordb Context pipeline With LLM output

In [65]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke(prompt)
    return response.content

In [67]:
answer=rag_simple("How does the pipeline handle vouchers that are deleted in Tally, and why can’t the AlterID filter detect them??",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'How does the pipeline handle vouchers that are deleted in Tally, and why can’t the AlterID filter detect them??'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The pipeline removes deleted vouchers with a separate nightly job: it requests the complete list of voucher GUIDs from Tally, compares that list to the GUIDs stored in the database, and deletes any records that no longer appear in Tally.

The regular incremental extraction can’t see deletions because Tally does not emit a record for a voucher that has been removed, and it does not generate a new AlterID for that event. Since the AlterID filter only returns rows whose $AlterID is greater than the last processed value, a missing voucher never produces a higher AlterID, so the filter cannot detect the deletion.


### Enhanced RAG Pipeline Features

In [75]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke(prompt)
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Tally", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Tally'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.57it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Tally (specifically Tally Prime) is an accounting/ERP software package that records financial transactions—vouchers, ledgers, trial balances, etc.—and provides a built‑in HTTP/XML server for programmatic data extraction.
Sources: [{'source': 'Tally Prime to Excel Reporting Pipeline.pdf', 'page': 2, 'score': 0.31823432445526123, 'preview': 'Amounts are normalised on load (debit positive, credit negative) and verified against a known \nvoucher. \n• Cancelled vouchers are kept, flagged, and excluded in the views, preserving the audit trail. \n \nStep 3: Extract Data from Tally \nTallyPrime has a built-in HTTP and XML server. The agent sends a...'}, {'source': 'Tally Prime to Excel Reporting Pipeline.pdf', 'page': 2, 'score': 0.31823432445526123, 'preview': 'Amounts are normalised on load (debit positive, credit negative) and verified against a known \nvoucher. \n• Cancelled vouchers are kept, flagged, and excluded in the views, preserving the audit trail. \n \nStep 3: Extract Data

In [81]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke(prompt)
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke(summary_prompt)
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Tally Prime", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Tally Prime'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.33it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Amounts are normalised on load (debit positive, credit negative) and verified against a known 
voucher. 
• Cancelled vouchers are kept, flagged, and excluded in the views, preserving the audit trail. 
 
Step 3: Extract Data from Tally 
TallyPrime has 

a built-in HTTP and XML server. The agent sends a request defined in Tally Definition 
Language (TDL) and receives only the fields it needs. 
Enabling the server: in TallyPrime, go to F1 (Help), then Settings, then Connectivity, then Client/Server 
configuration. Set TallyPrime to act as Server (or Both) on port 9000. The company must be open. 
Incremental extraction 
Every create, edit or delete in Tally increases its AlterID counter. The agent stores the last AlterID it 
processed and asks Tally only for records above that value. This keeps each run fast, typically seconds, 
instead of re-reading the whole company. 
Collection: Voucher 
Filter:     $AlterID > {last_alter_id}

Amounts are normalised on load (debit positive, credit negative) and verified against a known 
voucher. 
• Cancelled vouchers are kept, flagged, and excluded in the views, preserving the audit trail. 
 
Step 3: Extract Data from Tally 
TallyPrime has a built-in HTTP and XML server. The agent sends a request defi